<a href="https://colab.research.google.com/github/FlemingJohn/LLM/blob/main/LLM_Fine_Tunnig_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install datasets pandas torch transformers[torch] python-dotenv  peft

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
huggingface_token = os.getenv("HuggingFace_TOKEN")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pading_side = "right"
model = AutoModelForCausalLM.from_pretrained(model_name)
print(model)


In [ ]:
text = "Who is Fleming John"
inputs = tokenizer(text, return_tensors="pt")

output = model.generate(inputs.input_ids, max_new_tokens = 100)
print(tokenizer.decode(output[0],skip_special_tokens=True))

In [ ]:
from datasets import load_dataset

raw_data = load_dataset("/datasetpath",split = train"train[:1000]")
data = raw_data.train_test_split(train_size=0.95)
data

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
def pre_process_batch(batch):
  return tokenizer(batch["text"], truncation = True, padding = True, max_length = 200)
tokenized_dataset = data.map(
    pre_process_batch,
    batched = True,
    batch_size = 4
    remove_columns = data["train"].column_names
)
print(tokenized_dataset)


In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModelling(tokenizer=tokenizer, mlm=False)
data_collator


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.1,
    bias = "none",
    task_type = TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
model.train()
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr = 1e-5)
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir ="/.output",
    evaluvation_strategy = "epoch",
    save_steps = 500
    learning_rate = 1e-5,
    weight_decay = 0.01,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 2,
    logging_steps = 50,
    logging_dir ="./logs",
    resume_from_checkpoint = True
)

trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["test"],
    args = training_args,
    data_collator = data_collator
)
trainer.train()